# Ma Soi — Behavior Cloning locally (P0)

Train BC with P0 flags from the encoded dataset in `.tmp/enc` (§17).
The Colab version is `train_bc_colab.ipynb` — this one is local, no Drive/GPU.

## Before running

- Dataset: `.tmp/enc` (checked in section 1). If missing:
  `npm run ai:encode -- --in .tmp/bc/trajectories.jsonl --out .tmp/enc`.
- Any kernel works: torch/matplotlib run via the venv as subprocesses,
  the check cell only needs numpy (system python has it).
- CPU-only: several times slower than a Colab T4. Small net (96K params) —
  40 epochs still finish within a session, just let it run.
- Boundary (§39) unchanged: numbers from `.bin` only, no trajectory parsing.

## 0. Environment

Locate the repo root (open the notebook from the repo root or anywhere below it),
the venv and the dataset. If this fails, nothing below can run.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

_here = Path.cwd()
# Walk up from the notebook dir (VS Code sets cwd to ai-training/colab/)
# until something looks like the repo root.
ROOT = None
for _p in [_here, *_here.parents]:
    if (_p / 'ai-training' / 'masoi_training').exists():
        ROOT = _p
        break
if ROOT is None:
    raise SystemExit(f'cannot find repo root from {Path.cwd()} - open the notebook from the repo root')
VENV_PY = ROOT / 'ai-training' / '.venv' / 'Scripts' / 'python.exe'
TRAIN_DIR = ROOT / 'ai-training'
# Subprocesses inherit the kernel environment - notebook kernels set
# MPLBACKEND=module://matplotlib_inline... which the venv matplotlib lacks.
# Pin it here: every venv call uses the Agg backend (file output, no display).
VENV_ENV = {**os.environ, 'MPLBACKEND': 'Agg'}
DATA = ROOT / '.tmp' / 'enc-m'
SMOKE = ROOT / '.tmp' / 'smoke-local'
OUT = ROOT / '.tmp' / 'model-m'
print('ROOT ', ROOT)
print('DATA ', DATA, '->', 'found' if (DATA / 'meta.json').exists() else 'MISSING')
print('venv ', VENV_PY, '->', 'found' if VENV_PY.exists() else 'MISSING')
if not VENV_PY.exists():
    raise SystemExit('missing venv - install per ai-training/README.md first')
if not (DATA / 'meta.json').exists():
    raise SystemExit('missing .tmp/enc - run ai:encode first')

# torch/matplotlib live in the venv only: ask via subprocess so any kernel works.
ver = subprocess.run(
    [str(VENV_PY), '-c',
     'import torch; print(torch.__version__, torch.cuda.is_available(), torch.get_num_threads())'],
    capture_output=True, text=True, cwd=str(TRAIN_DIR), env=VENV_ENV,
)
print('torch in venv (version, cuda, threads):', (ver.stdout or ver.stderr).strip())
mpl = subprocess.run(
    [str(VENV_PY), '-c', 'import matplotlib; print(matplotlib.__version__)'],
    capture_output=True, text=True, env=VENV_ENV,
)
print('matplotlib (venv):', (mpl.stdout or mpl.stderr).strip())
print('this kernel:', sys.executable)

## 1. Validate data before training

Three questions before spending a CPU session:

1. Do files match `meta.json`?
2. Do train/val/test sizes sum to the full set (§15)?
3. **Is every label LEGAL under its own mask**?

Same as the Colab version: check inside a function, return numbers not arrays,
so RAM is released before the train cell spawns its subprocess (~6 GB peak full package).

In [ ]:
import gc

sys.path.insert(0, str(TRAIN_DIR))
from masoi_training.data import action_distribution, load


def check(path):
    data = load(str(path))  # throws if a file mismatches meta.json

    sizes = {name: len(data.split(name)) for name in ('train', 'validation', 'test')}
    assert sum(sizes.values()) == len(data)
    assert data.masks[range(len(data)), data.actions].all(), 'labels point at ILLEGAL actions'
    assert set(data.rewards.tolist()) <= {-1.0, 1.0}
    assert data.optimal is not None, 'missing optimal.u8.bin - re-run ai:encode'

    return {
        'version': data.meta.get('datasetVersion'),
        'commit': (data.meta.get('gitCommit') or '')[:8],
        'rows': len(data),
        'obs': data.obs_size,
        'actions': data.action_size,
        'games': data.meta.get('games'),
        'sizes': sizes,
        'classes': len(action_distribution(data)),
        'scores': data.scores is not None,
    }


info = check(DATA)
gc.collect()

print('dataset  ', info['version'], 'commit', info['commit'])
print('rows     ', info['rows'], '| obs', info['obs'], '| actions', info['actions'])
print('games    ', info['games'])
print('split    ', info['sizes'])
print('action classes (§43):', info['classes'])
print('scores   ', 'present (only used with --distill-alpha > 0)' if info['scores'] else 'absent - not needed by default')
print('\nOK - ready to train. RAM was released before the train cell.')

## 2. Smoke test: 2 epochs

Catch config errors here in minutes instead of after a full CPU run.

In [ ]:
import shutil
import subprocess

shutil.rmtree(SMOKE, ignore_errors=True)
done = subprocess.run(
    [str(VENV_PY), '-m', 'masoi_training.train_bc',
     '--data', str(DATA), '--out', str(SMOKE), '--epochs', '2'],
    cwd=str(TRAIN_DIR), env=VENV_ENV,
)
if done.returncode != 0:
    raise SystemExit(f'smoke test failed (exit {done.returncode}) - read the log above')
print('smoke OK')

## 3. Full train (P0)

P0 flags on: `--optimizer adamw` + `--weight-decay 0.01` + `--scheduler cosine`
against overfit, `--grad-clip 1.0` against exploding steps, `--patience 5` to stop
early keeping the best epoch, `--init orthogonal` for a stable start. All recorded
in `trainingConfig` inside `metrics.json`.

`--batch-size 512` matches every other report in the repo — keep it comparable.

P1-1 opt-in (commented in CMD): `--activation silu` / `--norm layernorm` export
`masoi-mlp-2`, which the engine now runs. Try only after the default run converges.

In [ ]:
import subprocess
import shutil
import time

CMD = [
    str(VENV_PY), '-m', 'masoi_training.train_bc',
    '--data', str(DATA),
    '--out', str(OUT),
    '--epochs', '100',
    '--batch-size', '512',
    '--lr', '1e-3',
    '--hidden', '128',
    '--seed', '12345',
    '--model-id', 'policy-p0',
    '--optimizer', 'adamw', '--weight-decay', '0.01',
    '--scheduler', 'cosine', '--warmup-epochs', '1',
    '--grad-clip', '1.0', '--patience', '5',
    '--init', 'orthogonal',
    '--activation', 'silu',
    '--norm', 'layernorm',
    '--value-trunk', 'separate', '--value-weight', '0.5',
]
 
shutil.rmtree(OUT, ignore_errors=True)
start = time.time()
done = subprocess.run(CMD, cwd=str(TRAIN_DIR), env=VENV_ENV)
print(f'\ntotal {time.time() - start:.0f}s  | exit {done.returncode}')
if done.returncode != 0:
    raise SystemExit(f'train_bc failed (exit {done.returncode}). Read the log above.')

## 4. Read the results

The deciding number is **`metrics.test.agreementTieAware`** — tied-for-best vs the
teacher on unseen games. Not `agreement` (penalizes tied moves) and not loss (§17).
Dataset ceiling: **0.984**.

In [ ]:
import json
import pathlib

path = OUT / "metrics.json"
if not path.exists():
    raise SystemExit(
        f"no {path} - training cell (section 3) did NOT finish or failed.\n"
        "Go back to the training cell, read the final 'exit ...' line. Don't edit this cell."
    )
report = json.loads(path.read_text())
m = report["metrics"]

print(f"{'':<11} {'tieAware':>9} {'agreement':>10} {'top-2':>8}")
for name in ("train", "validation", "test"):
    e = m[name]
    print(f"{name:<11} {e.get('agreementTieAware'):>9} {e.get('agreement'):>10} {e.get('top2Agreement'):>8}")

# Diagnose on the SAME metric used for conclusions. Mixing an `agreement` gap with
# `agreementTieAware` conclusions mixes two scales.
gap = m["train"]["agreementTieAware"] - m["validation"]["agreementTieAware"]
print(f"\ntrain - val = {gap:+.4f}  ->", "OVERFIT" if gap > 0.05 else "no overfit")
print("best epoch:", report["bestEpoch"], "/", report["trainingConfig"]["epochs"])

print("\nBy decision type (worst first):")
for kind, score in sorted(m["test"].get("agreementByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")

print("\nBy role (worst first):")
for role, score in sorted(m["test"].get("agreementByRole", {}).items(), key=lambda kv: kv[1]):
    print(f"  {role:<18} {score}")

print("\nCalibration (ECE - lower is better, > 0.1 is overconfident):")
print(f"  test ece: {m['test'].get('ece')}")
for kind, score in sorted(m["test"].get("eceByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")

In [ ]:
# Plot via the venv (the kernel may lack matplotlib): render PNG, then display.
import subprocess
import tempfile
from pathlib import Path as _P

PLOT_SRC = '''
import json, sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
out = sys.argv[1]
report = json.load(open(out + '/metrics.json', encoding='utf8'))
history = report['history']
epochs = [row['epoch'] for row in history]
figure, left = plt.subplots(figsize=(8, 4))
left.plot(epochs, [row['trainLoss'] for row in history], label='train loss')
left.set_xlabel('epoch')
left.set_ylabel('loss')
right = left.twinx()
right.plot(
    epochs,
    [r['val_agreementTieAware'] if r.get('val_agreementTieAware') is not None else r.get('val_agreement') for r in history],
    color='tab:orange',
    label='val agreement',
)
right.set_ylabel('agreement')
figure.legend(loc='upper right')
plt.title('Behavior cloning: falling loss is NOT enough, agreement must rise')
plt.savefig(out + '/history.png', dpi=100)
print('plotted', out + '/history.png')
'''

with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False, encoding='utf8') as f:
    f.write(PLOT_SRC)
    _script = f.name
_done = subprocess.run([str(VENV_PY), _script, str(OUT)], cwd=str(TRAIN_DIR), env=VENV_ENV)
_P(_script).unlink()
if _done.returncode != 0:
    raise SystemExit(f'plot failed (exit {_done.returncode})')
from IPython.display import Image, display

display(Image(str(OUT / 'history.png')))

## 5. Benchmark before using

Compare against the old champion (`.tmp/bc/model-v001`, test tieAware 0.9659):

```powershell
npm run ai:benchmark -- --model .tmp/model-local/model.weights.json --setups baseline,village,wolves,all,teacher
```

Keep `metrics.json` next to `model.weights.json` (§46).
NEVER overwrite `apps/server/assets/models/` — that's the model shipping in the image.

In [ ]:
for name in ('model.pt', 'model.weights.json', 'metrics.json', 'model.onnx'):
    p = OUT / name
    print(f'{name:<20}', f'{p.stat().st_size / 1e6:.1f} MB' if p.exists() else 'MISSING')

## Reading the numbers (P0)

| tieAware / ceiling (0.984) | Meaning | Do |
|---|---|---|
| < 0.35 | Barely learned anything | See below |
| 0.35 – 0.70 | Trend learned, bot not replicated | Raise `--epochs` (patience stops on its own) |
| > 0.75 | Baseline replicated well | §17 cleared, RL is on the table |

`train - val > +0.05` is overfit — fewer epochs, more `--weight-decay`, never `--hidden`.
`ece` lower is better: above 0.1 is overconfident, and PPO sampling at T=1 will miss
more often than agreement suggests. `bestEpoch` names the best epoch — not the last one.

Don't seed-shop. Changing `--seed` tests stability;
if two seeds disagree widely, neither number concludes anything.

## 6. RL: PPO from the BC clone (spec 2026-09-17)

Goal: each side **stronger than the heuristic** (≥ +2 points over `village-bc-0002`)
without pushing whole-table balance further from 50 %. Spec:
`docs/superpowers/specs/2026-09-17-rl-ppo-from-bc-design.md`, plan:
`docs/superpowers/plans/2026-09-17-rl-ppo-from-bc.md`.

**Needs plan Tasks 0–4 merged first** (section 6.0 checks it): the FINAL_VOTE
learned-pick fix, `selfplay --learned-decisions`, and the two new promotion gates
in `rl_loop.py`. Without them rollouts silently skip the trial and the gates don't exist.

**v2 (2026-09-17):** v1 made the village side WORSE (−7.0 after 10 iterations):
the untrained NIGHT decisions drifted. Fixed in `train_ppo`/`rl_loop` (entropy and KL
gate on trained rows only, KL anchor to the official champion `--anchor-kl 0.1`,
restart from the champion after a rejected benchmark). All run folders are now
`*-v2` so nothing resumes the v1 state. 6.1 also stops if an untrained decision
keeps < 97 % of its argmax.

| Section | What | CPU estimate |
|---|---|---|
| 6.1 | Pilot: 2 iterations × 600 games, village | ~45 min |
| 6.2 | Village: 10 iterations × 3000 games | ~3 h |
| 6.3 | Wolves: 10 iterations × 3000 games | ~3 h |
| 6.4 | Night only (optional) | ~1.5 h / side |
| 6.5 | Confirm on fresh seeds + verdict | ~40 min |

- **Resumable:** interrupt the kernel or shut the machine; re-run the SAME cell and
  `rl_loop` skips every finished step (`.done` markers). Don't delete `.tmp/rl/`.
- Output streams here AND into `.tmp/rl/<run>.log`.
- Rollout is TypeScript on CPU; a GPU would not shorten most of the run.
- Never overwrite `apps/server/assets/models/`. Packaging (6.5 verdict PASS) is done
  afterwards by Claude from the printed candidate path.

In [ ]:
import json
import shutil
import subprocess
import time

NPM = shutil.which('npm') or 'npm'
NPX = shutil.which('npx') or 'npx'
RL = ROOT / '.tmp' / 'rl'
CHAMPION0 = ROOT / 'apps' / 'server' / 'assets' / 'models' / 'village-bc-0002.weights.json'
# Spec D5: night stays out of the policy loss in stages 1-2.
NO_NIGHT = 'vote,final_vote,hunter_shot'
COMMON = [
    '--temperature', '1', '--lr', '1e-4', '--target-kl', '0.01',
    '--shaping-alpha', '1', '--baseline', 'role', '--bench-every', '5',
]
RL_ENV = {**VENV_ENV, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8'}


def run_logged(cmd, log, cwd=TRAIN_DIR):
    # Plain subprocess.run sends child output to the kernel console, not this
    # notebook (section 3 only showed 'total ...'). Stream it here and to a log.
    log.parent.mkdir(parents=True, exist_ok=True)
    start = time.time()
    with log.open('a', encoding='utf8') as f:
        f.write(f"\n$ {' '.join(map(str, cmd))}\n")
        with subprocess.Popen(
            [str(c) for c in cmd], cwd=str(cwd), env=RL_ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf8', errors='replace',
        ) as proc:
            for line in proc.stdout:
                print(line, end='')
                f.write(line)
    print(f'\n{time.time() - start:.0f}s | exit {proc.returncode}')
    if proc.returncode != 0:
        raise SystemExit(f'failed (exit {proc.returncode}) - full log: {log}')


def rl_loop(name, champion, side, iterations, games, extra=()):
    out = RL / name
    run_logged(
        [VENV_PY, 'rl_loop.py', '--champion', champion, '--side', side,
         '--iterations', iterations, '--games', games, '--out', out, *COMMON, *extra],
        RL / f'{name}.log',
    )
    return out


def latest_champion(out):
    # champion-0000 is the starting copy; anything after it was PROMOTED.
    # Read champions/ rather than state['champion']: with --bench-every 5 the
    # state can point at an unbenched challenger.
    files = sorted((out / 'champions').glob('champion-*.weights.json'))
    return files[-1], len(files) > 1


def show_state(out):
    state = json.loads((out / 'state.json').read_text(encoding='utf8'))
    print(f"{'iter':>4} {'score':>7} {'confirm':>8} {'other':>7} {'imbal':>6}")
    for row in state['scores']:
        print(f"{row['iteration']:>4} {row['score']:>+7.2f} {row.get('confirmScore', float('nan')):>+8.2f} "
              f"{row.get('otherSide', float('nan')):>+7.2f} {row.get('imbalance', float('nan')):>6.2f}")
    print(f"champion score {state['championScore']:+.2f} | other {state.get('championOther')} "
          f"| imbalance {state.get('championImbalance')}")
    champion, promoted = latest_champion(out)
    print('latest champion:', champion.name, '| PROMOTED' if promoted else '| never promoted')
    return champion, promoted


print('RL dir     ', RL)
print('champion-0 ', CHAMPION0, '->', 'found' if CHAMPION0.exists() else 'MISSING')

### 6.0 Prerequisites

Builds the TS packages and checks that plan Tasks 0–4 are in this checkout.
If it stops, merge/pull those tasks first — don't start a 3-hour run on old code.

In [ ]:
subprocess.run([NPM, 'run', 'build:deps'], cwd=str(ROOT), check=True, capture_output=True)

loop_help = subprocess.run([str(VENV_PY), 'rl_loop.py', '--help'], cwd=str(TRAIN_DIR), env=RL_ENV,
                           capture_output=True, text=True, encoding='utf8').stdout
selfplay_help = subprocess.run([NPX, 'tsx', 'apps/server/scripts/selfplay.ts', '--help'], cwd=str(ROOT),
                               capture_output=True, text=True, encoding='utf8').stdout
missing = [flag for flag in ('--learned-decisions', '--balance-slack', '--other-side-slack') if flag not in loop_help]
if '--learned-decisions' not in selfplay_help:
    missing.append('selfplay --learned-decisions')
if missing:
    raise SystemExit(f'missing {missing} - plan Tasks 0-4 are not in this checkout')
if not CHAMPION0.exists():
    raise SystemExit(f'missing {CHAMPION0}')
print('OK - pipeline has the four-decision flag and both new gates')

### 6.1 Pilot (~45 min)

Catches pipeline errors before an overnight run. The check cell below must print
`PILOT OK`; otherwise stop and send the log (`.tmp/rl/pilot-v2.log`) to Claude.

In [ ]:
PILOT = rl_loop('pilot-v2', CHAMPION0, 'village', 2, 600, ['--train-decisions', NO_NIGHT])

### 6.2 Village (v3: 20 iterations, ~6 h local / ~9 h Colab)

v2 showed PPO does learn — each 5-iteration block made the village **+0.7 to +1.3**
over the champion — but it could never promote: the loop restarted from the champion
after every rejected benchmark (so gains never added up), and the balance gate blocked
any village gain because the village already wins > 50 %. v3 (spec D11):

- `--bench-every 10`: 10 iterations accumulate before a benchmark judges them;
- `--balance-slack -1`: no balance gate while training ONE side — whole-table balance
  is checked once, at the end (6.5), after both sides trained.

Still required on BOTH seed sets: score ≥ champion + 2, wolves side not down > 1 point.

Reading `show_state`: `score` = village strength vs heuristic, `other` = wolves side,
`imbal` = |village win − 50 %| with the whole table on the model (reported, not gated).

In [ ]:
# Spec D11: side stages accumulate 10 iterations per benchmark; balance is gated in 6.5 only.
SIDE_STAGE = ['--bench-every', '10', '--balance-slack', '-1']
VILLAGE = rl_loop('bc-village-v3', CHAMPION0, 'village', 20, 3000, ['--train-decisions', NO_NIGHT, *SIDE_STAGE])
VILLAGE_CHAMP, village_ok = show_state(VILLAGE)

In [ ]:
# Spec: no promotion -> retry ONCE with --lr 3e-4; still nothing -> stop project A.
if not village_ok:
    VILLAGE = rl_loop('bc-village-v3-lr3', CHAMPION0, 'village', 20, 3000,
                      ['--train-decisions', NO_NIGHT, *SIDE_STAGE, '--lr', '3e-4'])
    VILLAGE_CHAMP, village_ok = show_state(VILLAGE)
if not village_ok:
    raise SystemExit('Village never promoted (also at lr 3e-4). Stop here and send both show_state tables to Claude '
                     '-> next is sub-project B (observation).')
print('village champion:', VILLAGE_CHAMP)

### 6.3 Wolves (v3: 20 iterations, ~6 h local / ~9 h Colab)

Starts from the village champion. `--shaping-decisions vote`: wolves' FINAL_VOTE
shaping signal was measured silent. Gate on the village side is now the "other side".
Same accumulate-then-judge rule as 6.2; training the wolves pulls the whole-table
village win rate back toward 50 %, which 6.5 then checks.

In [ ]:
WOLVES = rl_loop('bc-wolves-v3', VILLAGE_CHAMP, 'wolves', 20, 3000,
                 ['--train-decisions', NO_NIGHT, '--shaping-decisions', 'vote', *SIDE_STAGE])
WOLVES_CHAMP, wolves_ok = show_state(WOLVES)

In [ ]:
if not wolves_ok:
    WOLVES = rl_loop('bc-wolves-v3-lr3', VILLAGE_CHAMP, 'wolves', 20, 3000,
                     ['--train-decisions', NO_NIGHT, '--shaping-decisions', 'vote', *SIDE_STAGE, '--lr', '3e-4'])
    WOLVES_CHAMP, wolves_ok = show_state(WOLVES)
CANDIDATE = WOLVES_CHAMP if wolves_ok else VILLAGE_CHAMP
print('wolves promoted' if wolves_ok else 'wolves never promoted -> candidate stays the village champion; skip 6.4')
print('candidate so far:', CANDIDATE)

### 6.4 Night only (optional, 10 iterations per side, ~3 h local each)

Runs only if BOTH 6.2 and 6.3 promoted. Old ablations showed night gradients flip
sign between seed sets; if neither side promotes here, the night bottleneck is the
observation (sub-project B), not more iterations.

In [ ]:
if village_ok and wolves_ok:
    NIGHT_V = rl_loop('bc-night-village-v3', CANDIDATE, 'village', 10, 3000,
                      ['--train-decisions', 'night', *SIDE_STAGE])
    night_v_champ, night_v_ok = show_state(NIGHT_V)
    if night_v_ok:
        CANDIDATE = night_v_champ
    NIGHT_W = rl_loop('bc-night-wolves-v3', CANDIDATE, 'wolves', 10, 3000,
                      ['--train-decisions', 'night', '--shaping-decisions', 'vote', *SIDE_STAGE])
    night_w_champ, night_w_ok = show_state(NIGHT_W)
    if night_w_ok:
        CANDIDATE = night_w_champ
    if not (night_v_ok or night_w_ok):
        print('night flat on both sides -> note for sub-project B')
else:
    print('skipped (needs both 6.2 and 6.3 promoted)')
print('candidate:', CANDIDATE)

### 6.5 Confirm on fresh seeds + verdict (~40 min)

Both models on seed `confirm-0917` (never used by any iteration), 5 × 300 games,
all five setups including `teacher`. Criteria from the spec:

1. every side that promoted in 6.2–6.4 is ≥ +2 over `village-bc-0002`;
2. no side below −1;
3. whole-table imbalance ≤ 6.8 (current 5.8 + 1) — the ONLY balance gate since v3;
4. zero rule violations.

Send the printed block (verdict + candidate path) to Claude: PASS → packaging as
`village-ppo-0001`; FAIL → report, production keeps `village-bc-0002`.

In [ ]:
BENCH = ['--games', '300', '--repeat', '5', '--seed', 'confirm-0917',
         '--setups', 'baseline,village,wolves,all,teacher',
         '--learned-decisions', 'vote,night,final,hunter']
BASE_JSON = RL / 'confirm-v3-bc0002.json'
CAND_JSON = RL / 'confirm-v3-candidate.json'
if not BASE_JSON.exists():
    run_logged([NPM, 'run', 'ai:benchmark', '--', '--model', CHAMPION0, *BENCH, '--out', BASE_JSON],
               RL / 'confirm.log', cwd=ROOT)
if not CAND_JSON.exists():
    run_logged([NPM, 'run', 'ai:benchmark', '--', '--model', CANDIDATE, *BENCH, '--out', CAND_JSON],
               RL / 'confirm.log', cwd=ROOT)

In [ ]:
from rl_loop import imbalance_of, score_of

promoted_sides = {'village'} | ({'wolves'} if wolves_ok else set())
delta = {side: score_of(CAND_JSON, side) - score_of(BASE_JSON, side) for side in ('village', 'wolves')}
imbalance = imbalance_of(CAND_JSON)
violations = sum(row['violations'] for row in json.loads(CAND_JSON.read_text(encoding='utf8'))['rows'])

checks = {
    'promoted sides >= +2': all(delta[s] >= 2.0 for s in promoted_sides),
    'no side below -1': all(d >= -1.0 for d in delta.values()),
    'imbalance <= 6.8': imbalance <= 6.8,
    'zero violations': violations == 0,
}
print('=' * 60)
for side, d in delta.items():
    print(f'{side:<8} delta vs bc-0002 {d:+.2f}' + ('  (trained)' if side in promoted_sides else ''))
print(f'imbalance {imbalance:.2f} | violations {violations}')
for name, ok in checks.items():
    print(f"  [{'x' if ok else ' '}] {name}")
print('VERDICT:', 'PASS' if all(checks.values()) else 'FAIL')
print('candidate:', CANDIDATE)
print('=' * 60)